# Amis Rewire: Colab A100 Runner

Run cells from top to bottom on a fresh Google Colab A100 runtime. This notebook is a runner only; the implementation remains in the repository. The experiment executes the focused 6-run protocol: full 4-condition ablation on mt5-small (Baseline, CLRR-Enc, JEPA, JEPA+CLRR-Enc) and cross-validation on mbart-large-50 (Baseline vs JEPA+CLRR-Enc) with seed 42. Checkpoints are backed up independently after every saved epoch.

**Data:** [Zheng et al. (2022)](https://aclanthology.org/2022.nlp4dh-1.11/) Amis-Mandarin parallel corpus (5,751 sentences).
Hosted on [Google Drive](https://drive.google.com/drive/folders/1W-glGBpCz9R16Oy-P96jdK7YGSVuvdg2).

In [ ]:
# Cell 1 - Clone repository & set working directory
REPO_URL = 'https://github.com/HeyDunaX/CLRR.git'
REPO_DIR = '/content/CLRR'

from pathlib import Path
import os
import subprocess

if not Path(REPO_DIR).exists():
    print(f'Cloning {REPO_URL} into {REPO_DIR}...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Directory {REPO_DIR} already exists. Pulling latest commits...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

In [ ]:
# Cell 2 — Verify GPU: this notebook is optimized for an A100 (compute >= 8.0)
import subprocess
import torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'CUDA is unavailable. Please select a GPU runtime (preferably A100).'
major, minor = torch.cuda.get_device_capability(0)
device_name = torch.cuda.get_device_name(0)
print(device_name, 'PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, f'compute {major}.{minor}')
if major < 8:
    print('WARNING: GPU compute capability is < 8.0. BF16 acceleration requires Ampere (compute >= 8.0).')

In [ ]:
# Cell 3 — Checkpoint backup configuration
# Recommended: use a private Hugging Face repo or Google Drive for backups.
from pathlib import Path
import os

USE_DRIVE_BACKUP = False
BACKUP_DIR = '/content/amis_rewire_backups'
if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/amis_rewire_backups'
Path(BACKUP_DIR).mkdir(parents=True, exist_ok=True)

# Optional: Set HF_BACKUP_REPO and Colab Secret HF_TOKEN for automatic remote uploads
HF_BACKUP_REPO = os.environ.get('HF_BACKUP_REPO', '')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
print('Checkpoint backups:', BACKUP_DIR)
print('Hugging Face remote backup:', HF_BACKUP_REPO or 'DISABLED (optional)')

In [ ]:
# Cell 4 — Install dependencies (keeps Colab's CUDA-compatible torch)
import subprocess
subprocess.run(['bash', 'scripts/setup_colab.sh'], check=True)

## Dataset Verification & Preparation

The 5,751 parallel sentences from [Zheng et al. (2022)](https://aclanthology.org/2022.nlp4dh-1.11/) are located in `data/processed/{train,validation,test}.csv`.
If already present in the repository, this step confirms data readiness. Otherwise, it automatically downloads and unpacks `parallel.zip` from Google Drive.

In [ ]:
# Cell 5 — Verify or download dataset -> data/processed/*.csv
import zipfile
import csv
import subprocess
from pathlib import Path

OUTPUT_DIR = Path('data/processed')
ZIP_PATH = Path('parallel.zip')
DRIVE_URL = 'https://drive.google.com/drive/folders/1W-glGBpCz9R16Oy-P96jdK7YGSVuvdg2'
SPLIT_MAP = {
    'train':      ('parallel-data/ami.train', 'parallel-data/cmn.train'),
    'validation': ('parallel-data/ami.dev',   'parallel-data/cmn.dev'),
    'test':       ('parallel-data/ami.test',  'parallel-data/cmn.test'),
}

# Check if processed CSVs already exist in the repository
if (OUTPUT_DIR / 'train.csv').exists() and (OUTPUT_DIR / 'validation.csv').exists() and (OUTPUT_DIR / 'test.csv').exists():
    print('Processed dataset found directly in repository:')
    for split in ['train', 'validation', 'test']:
        with open(OUTPUT_DIR / f'{split}.csv', encoding='utf-8') as f:
            count = sum(1 for _ in f) - 1
        print(f'  {split}: {count} pairs')
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        print('Dataset not found locally. Attempting download from Google Drive...')
        try:
            subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
            subprocess.run(['gdown', '--folder', DRIVE_URL, '-O', 'drive_data'], check=False)
            for p in Path('drive_data').rglob('*.zip'):
                ZIP_PATH = p
                break
        except Exception as e:
            print('Notice during download attempt:', e)

    assert ZIP_PATH.exists() or (OUTPUT_DIR / 'train.csv').exists(), (
        f'Dataset not found. Please ensure data/processed/ exists or download parallel.zip from {DRIVE_URL}.'
    )

    if not (OUTPUT_DIR / 'train.csv').exists():
        with zipfile.ZipFile(ZIP_PATH) as zf:
            for split, (ami_name, cmn_name) in SPLIT_MAP.items():
                ami_lines = [l.strip() for l in zf.open(ami_name).read().decode('utf-8').splitlines() if l.strip()]
                cmn_lines = [l.strip() for l in zf.open(cmn_name).read().decode('utf-8').splitlines() if l.strip()]
                assert len(ami_lines) == len(cmn_lines), f'Line mismatch for {split}'
                out_path = OUTPUT_DIR / f'{split}.csv'
                with open(out_path, 'w', encoding='utf-8', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(['source', 'target'])
                    for ami, cmn in zip(ami_lines, cmn_lines):
                        writer.writerow([ami, cmn])
                print(f'{split}: {len(ami_lines)} examples -> {out_path}')

print('Data ready.')

In [ ]:
# Cell 6 — Smoke test: short mT5 jepa-clrr-enc run (verifies model hooks & evaluation pipeline)
import subprocess
subprocess.run([
    'python', '-m', 'amis_rewire.train',
    '--model', 'mt5-small', '--method', 'jepa-clrr-enc',
    '--rewire-stack', 'encoder', '--jepa-weight', '0.1',
    '--data-dir', 'data/processed', '--output-dir', 'outputs_smoke', '--run-name', 'smoke',
    '--seed', '42', '--num-train-epochs', '1', '--early-stopping-patience', '1',
    '--per-device-train-batch-size', '8', '--per-device-eval-batch-size', '8',
    '--gradient-accumulation-steps', '1', '--max-source-length', '128', '--max-target-length', '128',
    '--num-beams', '1', '--bf16', '--gradient-checkpointing', '--dataloader-num-workers', '2'
], check=True)
print('Smoke test passed successfully!')

In [ ]:
# Cell 7 - Full protocol: 6 focused runs (mt5-small 4 ablation conditions + mbart-large-50 2 runs, seed=42)
# Completed runs are skipped; interrupted runs resume automatically from latest checkpoint.
import os
import subprocess

os.environ['BACKUP_DIR'] = BACKUP_DIR
if HF_BACKUP_REPO:
    os.environ['HF_BACKUP_REPO'] = HF_BACKUP_REPO
subprocess.run(['bash', 'scripts/run_all_models.sh'], check=True)

In [ ]:
# Cell 8 — Aggregate results into a summary table
import subprocess
import pandas as pd
from IPython.display import display

subprocess.run([
    'python', 'scripts/summarize_results.py', '--outputs-dir', 'outputs',
    '--save', 'results_summary.csv'
], check=True)
display(pd.read_csv('results_summary.csv'))

In [ ]:
# Cell 9 — Archive outputs for download
import shutil
outputs_archive = shutil.make_archive('/content/amis_rewire_outputs', 'zip', root_dir='outputs')
backups_archive = shutil.make_archive('/content/amis_rewire_backups', 'zip', root_dir=BACKUP_DIR)
print('Outputs saved:', outputs_archive)
print('Backups saved:', backups_archive)